In [1]:
from pytential import sympy_pytential, min_pytential, quad_pytential
import numpy as np
from sympy import log, symbols
import plotly.graph_objects as go
from plotly.subplots import make_subplots

This file demonstrates how to create, combine, and manipulate pytentials.  Equibilriation is used to reduce free variables. 

# Create phases

Create two binary ideal solutions, using the penalization method to affect the lattice constraint via the elastic energy. 

In [2]:
c0a, c1a, c0b, c1b, Va, Vb = symbols('c0a, c1a, c0b, c1b, Va, Vb')

In [3]:
RT = 8.134*300
kappa = 100000
fa_sp = c0a*RT*(1+log(c0a/(c0a+c1a))) + c1a*RT*(0+log(c1a/(c0a+c1a)))+(c0a+c1a)*kappa/2*(log(Va/(c0a+c1a)))**2
fb_sp = c0b*RT*(0+log(c0b/(c0b+c1b))) + c1b*RT*(1+log(c1b/(c0b+c1b)))+(c0b+c1b)*kappa/2*(log(Vb/(c0b+c1b)))**2

In [4]:
# Build the pytentials from the sympy expressions
fa = sympy_pytential(fa_sp)
fb = sympy_pytential(fb_sp)
print(fa)

x = ['Va', 'c0a', 'c1a']

f(x) = 2440.2*c0a*(log(c0a/(c0a + c1a)) + 1) + 2440.2*c1a*log(c1a/(c0a + c1a)) + (50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))**2

f'(x)= [2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/Va, -2440.2*c1a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 50000*log(Va/(c0a + c1a))**2 + 2440.2*log(c0a/(c0a + c1a)) + 2440.2 - 2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/(c0a + c1a), -2440.2*c0a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c1a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 50000*log(Va/(c0a + c1a))**2 + 2440.2*log(c1a/(c0a + c1a)) - 2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/(c0a + c1a)]

f"(x)= [[-2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/Va**2 + 2*(50000*c0a + 50000*c1a)/Va**2, 100000*log(Va/(c0a + c1a))/Va - 2*(50000*c0a + 50000*c1a)/(Va*(c0a + c1a)), 100000*log(Va/(c0a + c1a))/Va - 2*(50000*c0a + 50000*c1a)/(Va*(c0a + c1a))], [100000*log(Va/(c0a + c1a))/Va - 2*(50000*c0a + 50000*c1a)/(Va*(c0a + c1a)), -2440.2*c0a/(c0a + c1a)**2 + 2440.2*c1a/

The elastic strain relaxes the lattice constraint. For reference, we can visualize $f^a$ and $f^b$ assuming the lattice constraint still holds; $c_0^a+c_1^a = V^a = 1$. 

In [5]:
x_values = np.linspace(0.001, .999, 100)
Fa = fa(c0a=x_values, c1a=1-x_values, Va = 1)
Fb = fb(c0b=x_values, c1b=1-x_values, Vb = 1)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_values, y=Fa, mode='lines', name='fa'))
fig.add_trace(go.Scatter(x=x_values, y=Fb, mode='lines', name='fb'))
fig.update_layout(
    xaxis_title='x',
    yaxis_title='Energy',
    title='fa and fb',
    legend_title='Function'
)
fig.show()

Now we examine the complete space of $f^a$ and $f^b$ and show the previous curves as the valley in the surface: 

In [6]:
# Create meshgrid
X, Y = np.meshgrid(x_values, x_values)

# Calculate Z values for fa and fb
Za = fa(Va=1, c0a=X.ravel(), c1a=Y.ravel()).reshape(X.shape)
Zb = fb(Vb=1, c0b=X.ravel(), c1b=Y.ravel()).reshape(X.shape)

# Line values for the constraint c0a + c1a = 1 (i.e., c1a = 1 - c0a)
fa_line = Fa
fb_line = Fb

# Create subplots for side-by-side surfaces
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('fa(Va=1)', 'fb(Vb=1)'),
    specs=[[{'type': 'surface'}, {'type': 'surface'}]]
)

# Add surface traces
fig.add_trace(go.Surface(z=Za, x=x_values, y=x_values, name='fa surface', showscale=False), row=1, col=1)
fig.add_trace(go.Surface(z=Zb, x=x_values, y=x_values, name='fb surface', showscale=False), row=1, col=2)

# Add line traces on top of each surface
fig.add_trace(go.Scatter3d(x=x_values, y=1-x_values, z=fa_line,
    mode='lines', line=dict(color='red', width=6), name='fa line'), row=1, col=1)

fig.add_trace(go.Scatter3d(x=x_values, y=1-x_values, z=fb_line,
    mode='lines', line=dict(color='blue', width=6), name='fb line'), row=1, col=2)

# Update layout for both subplots
fig.update_layout(
    title='Side-by-Side Surface Plots of fa and fb with Constraint Line',
    scene1=dict(xaxis_title='c0a', yaxis_title='c1a', zaxis_title='fa'),
    scene2=dict(xaxis_title='c0b', yaxis_title='c1b', zaxis_title='fb'),
)

fig.show()

# Combine functions

We now combine both functions into a composite pytential, and add constraints for the total of each species. 

Note the pytential takes c0 and c1 as arguments even though they only appear in the constraints. 

In [7]:
f = fa+fb
c0, c1 = symbols('c0, c1')
f = f.add_constraints_sym([c0a+c0b-c0, c1a+c1b-c1, Va+Vb-1]) 
print(f)

x = ['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']

f(x) = 2440.2*c0a*(log(c0a/(c0a + c1a)) + 1) + 2440.2*c0b*log(c0b/(c0b + c1b)) + 2440.2*c1a*log(c1a/(c0a + c1a)) + 2440.2*c1b*(log(c1b/(c0b + c1b)) + 1) + (50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))**2 + (50000*c0b + 50000*c1b)*log(Vb/(c0b + c1b))**2

f'(x)= [2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/Va, 2*(50000*c0b + 50000*c1b)*log(Vb/(c0b + c1b))/Vb, 0, -2440.2*c1a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c0a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 50000*log(Va/(c0a + c1a))**2 + 2440.2*log(c0a/(c0a + c1a)) + 2440.2 - 2*(50000*c0a + 50000*c1a)*log(Va/(c0a + c1a))/(c0a + c1a), -2440.2*c1b/(c0b + c1b) + 2440.2*(c0b + c1b)*(-c0b/(c0b + c1b)**2 + 1/(c0b + c1b)) + 50000*log(Vb/(c0b + c1b))**2 + 2440.2*log(c0b/(c0b + c1b)) - 2*(50000*c0b + 50000*c1b)*log(Vb/(c0b + c1b))/(c0b + c1b), 0, -2440.2*c0a/(c0a + c1a) + 2440.2*(c0a + c1a)*(-c1a/(c0a + c1a)**2 + 1/(c0a + c1a)) + 50000*log(Va/(c0a + c1a))**2 + 2440.2*log(c1a/(c0a + c1a)) - 2*(50000*c0

## Reduce dimensionality through minimization

$f(V^a, V^b, c_0^a,c_1^a,c_0^b,c_1^b, c_0, c_1)$ is now a function off all the relevant variables but to be useful we must reduce the dimensionality through applying constraints and minimization. 

We can now explore the minimizer capabilities. If we minimize over volume, we will find the lowest common tangent. If we keep it fixed at 0 or 1 we will recover the end members. 

In [8]:
f_min = min_pytential(f, ['c0', 'c1'])

In [9]:
x_values2 = np.linspace(0.05, .95, 10)
ym = f_min(c0=x_values2, c1 = 1-x_values2)

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:441: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:495: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds

c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\opti

In [10]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_values, y=Fa, mode='lines', name='fa', line=dict(color='green')))
fig.add_trace(go.Scatter(x=x_values, y=Fb, mode='lines', name='fb', line=dict(color='red')))
fig.add_trace(go.Scatter(x=x_values2, y=ym, mode='lines', name='f_min'))
fig.update_layout(
    xaxis_title='x',
    yaxis_title='Energy',
    title='fa and fb',
    legend_title='Function'
)
fig.show()

We can see what is happening by looking in 3D.

In [11]:
f_min_V = min_pytential(f, ['c0', 'c1', 'Va'])

# Create a meshgrid for the two arguments
X, Y = np.meshgrid(x_values2, x_values2)
Zmin = f_min_V(c0=X.ravel(), c1 = 1-X.ravel(), Va=Y.ravel()).reshape(X.shape)

In [12]:
fig = go.Figure(data=[go.Surface(z=Zmin, x=x_values2, y=x_values2)])

# Add a line plot for f_a at Va = 1
fig.add_trace(go.Scatter3d(
    x=x_values, y=0*x_values+1, z=Fa,
    mode='lines', name='fa', line=dict(color='green', width=6)
))
# Add a line plot for f_b at Va = 0
fig.add_trace(go.Scatter3d(
    x=x_values, y=0*x_values, z=Fb,
    mode='lines', name='fb', line=dict(color='red', width=6)
))
# Add labels and title
fig.update_layout(
    scene=dict(xaxis_title="c0", yaxis_title="Va", zaxis_title="f"),
)
fig.show()

Check the landscape of c0 vs c1 at a fixed V

In [13]:
Z = f_min_V(c0=X.ravel(), c1 = Y.ravel(), Va=.5).reshape(X.shape)

In [14]:
fig = go.Figure(data=[go.Surface(z=Z, x=x_values2, y=x_values2)])

# Add labels and title
fig.update_layout(
    title="Surface Plot of f_min2",
    scene=dict(
        xaxis_title="c0", yaxis_title="c1", zaxis_title="f_min2",
    ),
)
fig.show()

A convenient way to find the solubility limits (equilibrium state) is to minimize over ca and cb:

In [15]:
f0, y0 = f_min.min_fcn(np.array([[0.5, 0.5]]).T)
print(y0[0])

{'c0': 0.5, 'c1': 0.5, 'Va': 0.500001854828482, 'Vb': 0.499998145171518, 'c0a': 0.13447138796688796, 'c0b': 0.3655286120331121, 'c1a': 0.36553046686688784, 'c1b': 0.1344695331331121}


c:\Users\wellandm\AppData\Local\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:437: RuntimeWarning:

Values in x were outside bounds during a minimize step, clipping to bounds



Using the expansion point, we can build a pYtential as a quadratic expansion

In [16]:
fq = quad_pytential.from_homog_pyt(f, y0[0])
print(fq)
print(fq.hess())

x = ['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']

f(x) = 0.5*Va*(199999.258075594*Va - 199999.258073477*c0a - 199999.258073477*c1a) - 1.05875308522035e-6*Va + 0.5*Vb*(200000.74192991*Vb - 200000.741932028*c0b - 200000.741932028*c1b) + 1.05877528964721e-6*Vb + 0.5*c0a*(-199999.258073477*Va + 213265.487413438*c0a + 195118.876175954*c1a) - 764.417930735069*c0a + 0.5*c0b*(-200000.741932028*Vb + 201796.135023586*c0b + 195120.323829416*c1b) - 764.416639995029*c0b + 0.5*c1a*(-199999.258073477*Va + 195118.876175954*c0a + 201794.653494644*c1a) - 764.422360050353*c1a + 0.5*c1b*(-200000.741932028*Vb + 195120.323829416*c0b + 213267.185375998*c1b) - 764.433487370659*c1b

f'(x)= [199999.258075594*Va - 199999.258073477*c0a - 199999.258073477*c1a - 1.05875308522035e-6, 200000.74192991*Vb - 200000.741932028*c0b - 200000.741932028*c1b + 1.05877528964721e-6, 0, -199999.258073477*Va + 213265.487413438*c0a + 195118.876175954*c1a - 764.417930735069, -200000.741932028*Vb + 201796.135023586*c0b + 1951

In [17]:
fq2 = fq.reduce_by_eliminating_linear_constraints(vars_to_keep=['c0', 'c1', 'Va'])
print(fq2)

['Va', 'Vb', 'c0', 'c0a', 'c0b', 'c1', 'c1a', 'c1b']
['c0', 'c1', 'Va']
[2, 5, 0]
x = ['c0', 'c1', 'Va', 'c0b', 'c1b']

f(x) = 0.5*Va*(400000.000005504*Va - 199999.258073477*c0 + 400000.000005505*c0b - 199999.258073477*c1 + 400000.000005505*c1b) - 200000.741932028*Va + 0.5*c0*(-199999.258073477*Va + 213265.487413438*c0 - 213265.487413438*c0b + 195118.876175954*c1 - 195118.876175954*c1b) - 764.417930735069*c0 + 0.5*c0b*(400000.000005505*Va - 213265.487413438*c0 + 415061.622437024*c0b - 195118.876175954*c1 + 390239.20000537*c1b) - 200000.740641288*c0b + 0.5*c1*(-199999.258073477*Va + 195118.876175954*c0 - 195118.876175954*c0b + 201794.653494644*c1 - 201794.653494644*c1b) - 764.422360050353*c1 + 0.5*c1b*(400000.000005505*Va - 195118.876175954*c0 + 390239.20000537*c0b - 201794.653494644*c1 + 415061.838870641*c1b) - 200000.753059348*c1b + 100000.370966014

f'(x)= [-199999.258073477*Va + 213265.487413438*c0 - 213265.487413438*c0b + 195118.876175954*c1 - 195118.876175954*c1b - 764.41793073506

In [18]:
fq3 = fq2.reduce_uncontrained_by_minimization(vars_to_keep=['c0', 'c1', 'Va'])
print(fq3)

[0, 1, 2]
x = ['c0', 'c1', 'Va']

f(x) = 0.5*Va*(2633.01851319533*Va + 2848.8585297364*c0 - 2848.8585339993*c1) - 1316.50925449311*Va + 0.5*c0*(2848.8585297364*Va + 103082.39189882*c0 + 96917.6081011498*c1) - 102188.850434747*c0 + 0.5*c1*(-2848.8585339993*Va + 96917.6081011497*c0 + 103082.39189888*c1) - 99339.9919028737*c1 + 50329.127313091

f'(x)= [2848.8585297364*Va + 103082.39189882*c0 + 96917.6081011498*c1 - 102188.850434747, -2848.8585339993*Va + 96917.6081011498*c0 + 103082.39189888*c1 - 99339.9919028737, 2633.01851319533*Va + 2848.8585297364*c0 - 2848.8585339993*c1 - 1316.50925449311]

f"(x)= [[103082.391898820, 96917.6081011498, 2848.85852973640], [96917.6081011498, 103082.391898880, -2848.85853399930], [2848.85852973640, -2848.85853399930, 2633.01851319533]]


In [19]:
fig = go.Figure()
#data=[go.Surface(z=Zmin, x=x_values2, y=x_values2)])

# Calculate Z values for fq3
Zq3 = fq3(c0=X.ravel(), c1=(1-X).ravel(), Va=Y.ravel()).reshape(X.shape)

# Add the surface plot for fq3
fig.add_trace(go.Surface(z=Zq3, x=x_values2, y=x_values2, colorscale='Viridis', name='fq3'))


# Add a line plot for f_a at Va = 1
fig.add_trace(go.Scatter3d(
    x=x_values, y=0*x_values+1, z=Fa,
    mode='lines', name='fa', line=dict(color='green', width=6)
))
# Add a line plot for f_b at Va = 0
fig.add_trace(go.Scatter3d(
    x=x_values, y=0*x_values, z=Fb,
    mode='lines', name='fb', line=dict(color='red', width=6)
))
# Add labels and title
fig.update_layout(
    scene=dict(xaxis_title="c0", yaxis_title="Va", zaxis_title="f"),
)
fig.show()

In [20]:
fig = go.Figure()
#data=[go.Surface(z=Zmin, x=x_values2, y=x_values2)])

# Calculate Z values for fq3
Zqc3 = fq3(c0=X.ravel(), c1=Y.ravel(), Va=.5).reshape(X.shape)

# Add the surface plot for fq3
fig.add_trace(go.Surface(z=Zqc3, x=x_values2, y=x_values2, colorscale='Viridis', name='fqc3'))

fig.show()